In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
%run ./UDF/udf_silver_incremental_ingest

In [0]:


src_bronze_path = "/Volumes/data_governance/bronze_cost_monitoring/billing_pricing"
tgt_silver_table = "data_governance.silver_cost_monitoring.billing_pricing"



In [0]:
df=silver_incremental_ingest(src_bronze_path,tgt_silver_table)

In [0]:
if df.count()==0:
    dbutils.notebook.exit("No new records to load")
else:
    pass

In [0]:
df = df.select(
    col("account_id"),
    col("sku_name"),
    col("cloud"),
    col("currency_code"),
    col("usage_unit"),

    to_timestamp("price_start_time").alias("price_start_time"),

    col("pricing.default").cast("double").alias("default_price"),

    col("pricing.effective_list.default")
        .cast("double")
        .alias("effective_price"),

    col("pricing.promotional.default")
        .cast("double")
        .alias("promotional_price"),

    col("load_timestamp")
)


df = df.withColumn(
    "region",
    regexp_extract(col("sku_name"), "([A-Z]+_[A-Z]+)$", 1)
)


df = df.withColumn(
    "service_name",
    when(col("sku_name").contains("PRIVATE_CONNECTIVITY"), "Networking")
    .when(col("sku_name").contains("SQL"), "SQL Warehouse")
    .when(col("sku_name").contains("JOBS"), "Compute Jobs")
    .otherwise("Other")
)


df = df.dropDuplicates([
    "account_id",
    "sku_name",
    "price_start_time"
])

In [0]:
df.limit(20).display()

In [0]:
# Writing with Liquid Clustering
df.write\
 .format("delta")\
 .mode("append") \
 .partitionBy("cloud", "currency_code") \
 .saveAsTable(tgt_silver_table)

In [0]:
%sql
select * from data_governance.silver_cost_monitoring.billing_pricing;